In [1]:
from datasets import load_dataset


issues_dataset = load_dataset("lewtun/github-issues", split="train")
issues_dataset

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


datasets-issues-with-comments.jsonl: reconstructing file:   0%|          |  0.00B / 12.2MB            

datasets-issues-with-comments.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3019 [00:00<?, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 3019
})

In [2]:
# "The first order of business is to filter out the pull requests, as 
# these tend to be rarely used for answering user queries and will introduce noise in our search engine"
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

Filter:   0%|          | 0/3019 [00:00<?, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 808
})

In [3]:
# We can see that there are a lot of columns in our dataset, most of which we don’t need to build our search engine
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 808
})

In [4]:
# To create our embeddings we’ll augment each comment with the issue’s title and body, since these 
# fields often include useful contextual information
import pandas as pd
from typing import cast

issues_dataset.set_format("pandas")
df: pd.DataFrame = cast(pd.DataFrame, issues_dataset[:])

df["comments"][0].tolist()

['Cool, I think we can do both :)',
 '@lhoestq now the 2 are implemented.\r\n\r\nPlease note that for the the second protection, finally I have chosen to protect the master branch only from **merge commits** (see update comment above), so no need to disable/re-enable the protection on each release (direct commits, different from merge commits, can be pushed to the remote master branch; and eventually reverted without messing up the repo history).']

In [5]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(4)

,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,Protect master branch,"Cool, I think we can do both :)",After accidental merge commit (91c55355b634d0d...
1,https://github.com/huggingface/datasets/issues...,Protect master branch,@lhoestq now the 2 are implemented.\r\n\r\nPle...,After accidental merge commit (91c55355b634d0d...
2,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Hi ! I guess the caching mechanism should have...,## Describe the bug\r\nAfter upgrading to data...
3,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,"If it's easy enough to implement, then yes ple...",## Describe the bug\r\nAfter upgrading to data...


---

In [6]:
# Same stuff with dataset config, all in one place

from datasets import Dataset

comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset = comments_dataset.map(
    lambda x: {"comment_length": len(x["comments"].split())}
)
comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)

def concatenate_text(examples):
    return {
        "text": examples["title"]
        + " \n "
        + examples["body"]
        + " \n "
        + examples["comments"]
    }


comments_dataset = comments_dataset.map(concatenate_text)


Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2964 [00:00<?, ? examples/s]

Map:   0%|          | 0/2175 [00:00<?, ? examples/s]

In [7]:
import torch
from transformers import AutoTokenizer, AutoModel
from typing import Callable

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

BACKENDS: dict[str, Callable] = {
    "cpu": lambda: torch.device("cpu"),
    "cuda": lambda: torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"),
    "mps": lambda: torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu"),
}

device = BACKENDS["mps"]()  # pick your fighter here
model.to(device)

def cls_pooling(model_output) -> torch.Tensor:
    return model_output.last_hidden_state[:, 0]

def get_embeddings(text_list: list[str]) -> torch.Tensor:
    encoded_input = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
embedding = get_embeddings(comments_dataset["text"][0])
embedding.shape

torch.Size([1, 768])

In [9]:
import time
import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from pathlib import Path

texts = comments_dataset["text"]

def get_batch_embeddings(batch: list[str]) -> np.ndarray:
    encoded_input = tokenizer(
        batch,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}

    with torch.inference_mode():
        model_output = model(**encoded_input)

    return cls_pooling(model_output).cpu().numpy()

path = Path("comments_dataset_with_embeddings")
if path.exists():
    comments_dataset = Dataset.load_from_disk(path)
else: 
    loader = DataLoader(texts, batch_size=128, collate_fn=lambda x: x)

    batches: list[np.ndarray] = []
    t0 = time.perf_counter()
    for batch in tqdm(loader, desc="Embedding", unit="batch"):
        batches.append(get_batch_embeddings(batch))
    elapsed = time.perf_counter() - t0
    embeddings = np.concatenate(batches)
    print(f"{len(texts)} texts embedded in {elapsed:.2f}s ({len(texts)/elapsed:.1f} texts/s)")


In [10]:
from datasets import Dataset, List, Value

# datasets.add_column() can't infer a schema from a raw 2-D numpy array
# (ArrowInvalid: only handle 1-dimensional arrays), so route it through
# Dataset.from_dict, which handles numpy natively, then cast to float32.
if path.exists():
    comments_dataset = Dataset.load_from_disk(path)
else:
    embeddings_column = Dataset.from_dict({"embeddings": embeddings})["embeddings"]
    comments_dataset = comments_dataset.add_column("embeddings", embeddings_column)
    comments_dataset = comments_dataset.cast_column("embeddings", List(Value("float32")))
    comments_dataset.save_to_disk("comments_dataset_with_embeddings")


In [11]:
embeddings_dataset = comments_dataset


In [12]:
# FAISS crashes my ipy kernel on macOS (macOS + ARM + torch build configuration).

# I drop the FAISS, as index for ~2k documents is overkill anyway.
# Instead: Plain dot-product

# Plain dot-product - exact result
# FAISS - gives aproximate

import json
import numpy as np

doc_matrix = np.stack(embeddings_dataset["embeddings"]).astype(np.float32)
doc_matrix /= np.linalg.norm(doc_matrix, axis=1, keepdims=True) # Cos similarity

def search(question: str, k: int = 5) -> list[dict]:
    """Return top-k results as dicts with the tutorial's result fields:
    COMMENT, SCORE, TITLE, URL."""
    question_embedding = get_embeddings([question]).cpu().detach().numpy().astype(np.float32)[0]
    question_embedding /= np.linalg.norm(question_embedding)

    similarities = doc_matrix @ question_embedding # dot product
    top = np.argsort(-similarities)[:k]

    results = []
    for idx in top:
        row = embeddings_dataset[int(idx)]
        results.append({
            "COMMENT": row["comments"],
            "SCORE": float(similarities[idx]),
            "TITLE": row["title"],
            "URL": row["html_url"],
        })
    return results


In [13]:
example_queries = [
    "How can I load a dataset offline?",
    "How do I split a dataset into train and test sets?",
    "Cannot save dataset to disk, permission error",
]

for i, question in enumerate(example_queries, start=1):
    results = search(question, k=5)
    with open(f"search_results_query_{i}.json", "w") as f:
        json.dump({"query": question, "results": results}, f, indent=2, ensure_ascii=False)
    print(f"query {i} -> search_results_query_{i}.json ({len(results)} results)")


query 1 -> search_results_query_1.json (5 results)
query 2 -> search_results_query_2.json (5 results)
query 3 -> search_results_query_3.json (5 results)


In [14]:
import json
from pprint import pprint

with open("search_results_query_1.json") as f:
    data = json.load(f)

print("QUERY:", data["query"])
pprint(data["results"], width=100, sort_dicts=False)


QUERY: How can I load a dataset offline?
[{'COMMENT': 'here is my way to load a dataset offline, but it **requires** an online machine\r\n'
             '1. (online machine)\r\n'
             '```\r\n'
             'import datasets\r\n'
             'data = datasets.load_dataset(...)\r\n'
             'data.save_to_disk(/YOUR/DATASET/DIR)\r\n'
             '```\r\n'
             '2. copy the dir from online to the offline machine\r\n'
             '3. (offline machine)\r\n'
             '```\r\n'
             'import datasets\r\n'
             'data = datasets.load_from_disk(/SAVED/DATA/DIR)\r\n'
             '```\r\n'
             '\r\n'
             'HTH.',
  'SCORE': 0.7284624576568604,
  'TITLE': 'Discussion using datasets in offline mode',
  'URL': 'https://github.com/huggingface/datasets/issues/824'},
 {'COMMENT': '> here is my way to load a dataset offline, but it **requires** an online machine\n'
             '> \n'
             '> 1. (online machine)\n'
             '> \n'
   

In [15]:
import json
from pprint import pprint

with open("search_results_query_2.json") as f:
    data = json.load(f)

print("QUERY:", data["query"])
pprint(data["results"], width=100, sort_dicts=False)


QUERY: How do I split a dataset into train and test sets?
[{'COMMENT': "Currently you can indeed split a dataset using `ds_test = nlp.load_dataset('imdb, "
             "split='test[:5000]')` (works also with percentages).\r\n"
             '\r\n'
             "However right now we don't have a way to shuffle a dataset but we are thinking about "
             'it in the discussion in #166. Feel free to share your thoughts about it.\r\n'
             '\r\n'
             'One trick that you can do until we have a better solution is to shuffle and split '
             'the indices of your dataset:\r\n'
             '```python\r\n'
             'import nlp\r\n'
             'from sklearn.model_selection import train_test_split\r\n'
             '\r\n'
             "imdb = nlp.load_dataset('imbd', split='test')\r\n"
             'test_indices, val_indices = train_test_split(range(len(imdb)))\r\n'
             '```\r\n'
             '\r\n'
             'and then to iterate each split:\r\n'
 

In [16]:
import json
from pprint import pprint

with open("search_results_query_3.json") as f:
    data = json.load(f)

print("QUERY:", data["query"])
pprint(data["results"], width=100, sort_dicts=False)


QUERY: Cannot save dataset to disk, permission error
[{'COMMENT': "Hi ! That's because you are trying to overwrite a file that is already open and "
             'being used.\r\n'
             'Indeed `foo/dataset.arrow` is open and used by your `dataset` object.\r\n'
             '\r\n'
             'When you do `rename_column`, the resulting dataset reads the data from the same '
             'arrow file.\r\n'
             'In other cases like when using `map` on the other hand, the resulting dataset reads '
             'the data from another arrow file that is the result of the map transform.\r\n'
             '\r\n'
             'Therefore overwriting a dataset after `rename_column` is not possible, but it is '
             "possible after `map`, since `rename_column` doesn't switch to using another arrow "
             'file (the actual data stay the same).',
  'SCORE': 0.699334979057312,
  'TITLE': 'cannot save the dataset to disk after rename_column',
  'URL': 'https://github.c

## Analiza wyników

**Query 1: "How can I load a dataset offline?"**
Sukces. Wszystkie 5 trafień pochodzi z issue #824 "Discussion using datasets in offline mode". Najlepszy komentarz zawiera konkretne kroki: `save_to_disk` -> kopiowanie katalogu -> `load_from_disk`. Recall i precision@5 praktycznie 100%. Minus: wyniki dublują się na poziomie issue (5 komentarzy z jednego wątku).

**Query 2: "How do I split a dataset into train and test sets?"**
Większość trafień poprawna. Top 4 pochodzi z issue "documentation missing how to split a dataset" i zawiera rozwiązania: `split='test[:5000]'` oraz `train_test_split()`. Piąte trafienie (issue "[XGLUE] Adding new dataset") to pomyłka wynikająca z powierzchownego podobieństwa słów "train" i "dataset". Precision@5 około 0.8 - model nie jest w pełni odporny na płytkie podobieństwo leksykalne.

**Query 3: "Cannot save dataset to disk, permission error"**
Przypadek problematyczny. Wszystkie wyniki dotyczą `save_to_disk`, ale tylko trafienie 4 ("Windows Permission Error") dotyczy uprawnień. Wyniki 1-3 to inne błędy zapisu (nadpisanie pliku, refaktoryzacja). Model ignoruje negację ("cannot", "error") i dopasowuje ogólny temat zapisu na dysk zamiast konkretnej przyczyny błędu. Typowa słabość embeddingów w rozróżnianiu intencji i negacji.

### Zaobserwowane wzorce błędów
1. Quasi-duplikaty - jedno popularne issue zajmuje wiele miejsc w top-k przez osobne komentarze. Rozwiązaniem byłby post-filtr po ID issue.
2. Podobieństwo leksykalne - dokumenty nieistotne, ale zawierające słowa klucze (train, dataset), mogą wypierać trafne treści.
3. Brak rozumienia negacji i intencji - frazy "cannot" czy "permission error" są traktowane jako ogólny temat "saving to disk".
4. Obcinanie tekstu - limit 512 tokenów ucina końcówki długich komentarzy, gdzie często znajduje się właściwa odpowiedź.

### Dlaczego nie FAISS?
1. Nie ma opublikowanej wersji dla uv
2. FAISS jako indeks przybliżony nie ma większego sensu dla 2175 dokumentów - mnożenie macierzy jest dokładne i trwa poniżej 1 ms. Dodatkowo FAISS na tej maszynie wywala kernel Jupytera.
